# 02 — Chia chunk tài liệu



In [1]:
from pathlib import Path
import json
import re

ROOT = Path("..").resolve()
if not (ROOT / "knowledge").exists():
    ROOT = Path.cwd()
    if (ROOT / "vaxcare-ai" / "knowledge").exists():
        ROOT = ROOT / "vaxcare-ai"

CLEAN_DIR = ROOT / "knowledge" / "clean"
OUT_PATH = ROOT / "knowledge" / "chunks" / "chunks.jsonl"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

MAX_CHARS = 900
OVERLAP = 120

print("CLEAN =", CLEAN_DIR)
print("OUT   =", OUT_PATH)


CLEAN = D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\clean
OUT   = D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl


In [2]:
DIEU_RE = re.compile(r"(?=(?:^|\n)(Điều\s+\d+[^\n]*))", re.IGNORECASE)
MUC_RE = re.compile(r"(?=(?:^|\n)((?:Điều|Chương|Mục|Article)\s+[^\n]+))", re.IGNORECASE)


def parse_frontmatter(text: str):
    if not text.startswith("---"):
        return {}, text
    parts = text.split("---", 2)
    if len(parts) < 3:
        return {}, text
    meta = {}
    for line in parts[1].strip().splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip()
    return meta, parts[2].strip()


def split_by_dieu(body: str):
    matches = list(DIEU_RE.finditer(body))
    if len(matches) < 2:
        matches = list(MUC_RE.finditer(body))
    if not matches:
        return [("", body.strip())]
    chunks = []
    if matches[0].start() > 0:
        pre = body[: matches[0].start()].strip()
        if pre:
            chunks.append(("preamble", pre))
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(body)
        block = body[start:end].strip()
        title_line = block.split("\n", 1)[0].strip()
        chunks.append((title_line[:120], block))
    return chunks


def window_split(text: str, max_chars=MAX_CHARS, overlap=OVERLAP):
    text = text.strip()
    if len(text) <= max_chars:
        return [text] if text else []
    parts = []
    i = 0
    while i < len(text):
        parts.append(text[i : i + max_chars])
        i += max_chars - overlap
    return parts


In [3]:
records = []
for md in sorted(CLEAN_DIR.glob("*.md")):
    text = md.read_text(encoding="utf-8")
    meta, body = parse_frontmatter(text)
    doc_id = meta.get("doc_id") or md.stem
    sections = split_by_dieu(body)
    idx = 0
    for title, sec in sections:
        for piece in window_split(sec):
            if len(piece) < 40:
                continue
            records.append({
                "chunk_id": f"{doc_id}_{idx:04d}",
                "doc_id": doc_id,
                "title": title or doc_id,
                "text": piece,
                "source_file": meta.get("source_file", md.name),
                "tier": "A",
                "vaccine_ids": [],
                "page": None,
            })
            idx += 1
    print(f"[OK] {md.name}: {idx} chunks")

with OUT_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(records)} chunks -> {OUT_PATH}")
if records:
    print("Sample:", json.dumps(records[0], ensure_ascii=False)[:300], "...")


[OK] 6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em.md: 51 chunks


[OK] nd13-2024.md: 9 chunks
[OK] tt-10-2024.md: 13 chunks
Wrote 73 chunks -> D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl
Sample: {"chunk_id": "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em_0000", "doc_id": "6381586785791217441575-qđ-byt-về-việc-hướng-dẫn-khám-sàng-lọc-trước-tiêm-chủng-đối-với-trẻ-em", "title": "preamble", "text": "BỘ Y TẾ\nCỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM\nĐộc ...
